# Module 05: SimPy Digital Twin Evaluation Harness

The live queueing engine lives in `core/twin_sim.py` and runs inside the FastAPI server (`core/main.py`), not as a notebook artifact. This notebook is a sandbox for exercising it directly.

In [ ]:
!pip install -q simpy==4.1.1


## Standalone evaluation harness for the SimPy queueing engine

This notebook does **not** produce a `.pkl` - Module 05 *is* `core/twin_sim.py`,
which the FastAPI server imports directly (see `core/main.py`). This notebook
is a lightweight sandbox to sanity-check runway throughput and disruption
behaviour without spinning up the full FastAPI/WebSocket stack, e.g. when
tuning `BASE_FLIGHT_SPAWN_INTERVAL_S` in `core/config.py`.

In [ ]:
import sys
# When running locally (not Colab), point this at your cloned repo root:
# sys.path.insert(0, "/path/to/AeroTwin")

import simpy

try:
    from core.twin_sim import VadodaraAirport

    env = simpy.Environment()
    airport = VadodaraAirport(env)

    for i in range(6):
        env.process(airport.pushback_and_depart(f"6E-{100 + i}"))

    # Inject a runway closure partway through, exactly like the live /api/disrupt endpoint would
    def disruption_watcher():
        yield env.timeout(20)
        print(f"T={env.now:.1f}: injecting a 0.5-minute runway closure")
        airport.inject_disruption("runway_closure", duration_minutes=0.5)
    env.process(disruption_watcher())

    env.run(until=180)

    print("\nFinal flight states:")
    for fid, f in airport.flights.items():
        print(f"  {fid}: {f['status']} (risk={f['risk']})")
except ImportError as exc:
    print(f"core.twin_sim not importable in this environment ({exc}).")
    print("This notebook is meant to be run from the repo root, not Colab in isolation.")
